# 1. Environment Setup 
## Import Libraries 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

print("All Libraries loaded successfully")

ModuleNotFoundError: No module named 'matplotlib'

# 2. Load Data into MySQL & Python
## Connect Python to mySQL

In [ ]:
engine = create_engine('mysql+mysqlconnector://root:@localhost/md_water_services')
with engine.connect() as conn:
    print("MySQL connection successful!")


## Load Tables into Pandas DataFrame

In [ ]:
conn_direct = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="md_water_services"
)

water_source  = pd.read_sql("SELECT * FROM water_source",  conn_direct)
visits        = pd.read_sql("SELECT * FROM visits",        conn_direct)
location      = pd.read_sql("SELECT * FROM location",      conn_direct)
water_quality = pd.read_sql("SELECT * FROM water_quality", conn_direct)
well_poll     = pd.read_sql("SELECT * FROM well_pollution", conn_direct)
employee      = pd.read_sql("SELECT * FROM employee",      conn_direct)
global_wa     = pd.read_sql("SELECT * FROM global_water_access", conn_direct)

conn_direct.close()
print("All tables loaded successfully!")

## Look at each Table

In [ ]:
for name, df in [("water_source", water_source), ("visits", visits),
                 ("location", location), ("water_quality", water_quality)]:
    print(f"\n{'='*50}")
    print(f"TABLE: {name}")
    print(df.head(3))
    print(df.info())

# 3.Data Cleaning and Preprocessing
## 3.1 Check for Missing Values

In [ ]:
for name, df in [("water_source", water_source), ("visits", visits),
                 ("location", location), ("water_quality", water_quality),
                 ("well_pollution", well_poll)]:
    nulls = df.isnull().sum()
    if nulls.sum() > 0:
        print(f"\nNulls in {name}:")
        print(nulls[nulls > 0])

## 3.2 Check for Duplicate Records

In [ ]:
for name, df in [("water_source", water_source), ("visits", visits)]:
    dups = df.duplicated().sum()
    print(f"{name}: {dups} duplicate rows")

## 3.3 Check Water Source Types

In [2]:
# The data dictionary says type can be: tap_in_home, tap_in_home_broken,
# well, shared_tap, river
print("Water source types found:")
print(water_source['type_of_water_source'].value_counts())

# Check for unexpected values
valid_types = ['tap_in_home', 'tap_in_home_broken', 'well', 'shared_tap', 'river']
invalid = water_source[~water_source['type_of_water_source'].isin(valid_types)]
print(f"\nInvalid type entries: {len(invalid)}")

Water source types found:


NameError: name 'water_source' is not defined

## 3.4 Check Well Pollution Results

In [ ]:
print("Well pollution results:")
print(well_poll['results'].value_counts())

# Valid values should be: clean, dirty, biologically contaminated
print("\nPollutant ppm stats:")
print(well_poll['pollutant_ppm'].describe())

## 3.5 Check Queue Times

In [ ]:
# time_in_queue = minutes people wait — should not be negative
print("Queue time stats:")
print(visits['time_in_queue'].describe())

# Flag negative or unreasonably high values (e.g. > 600 min = 10 hours)
odd_queue = visits[(visits['time_in_queue'] < 0) | (visits['time_in_queue'] > 600)]
print(f"\nSuspect queue times: {len(odd_queue)}")

## 3.6 Create the Master Analysis DataFrame
###  Joining all tables into one clean dataframe for analysis

In [ ]:
# Join visits → water_source → location → water_quality
master = (
    visits
    .merge(water_source, on='source_id', how='left')
    .merge(location, on='location_id', how='left')
    .merge(water_quality, on='record_id', how='left')
)

# Add well pollution for wells only
master = master.merge(
    well_poll[['source_id','results','pollutant_ppm','biological']],
    on='source_id', how='left'
)

print(f"Master dataframe shape: {master.shape}")
print(master.head(3))

## 3.7 Create the Functionality Label 
### label for "functional" vs "non-functional" water Sources

In [3]:
def classify_status(row):
    """
    Rules based on water point type and quality:
    - tap_in_home = functional
    - tap_in_home_broken = non_functional
    - river = non_functional (unsafe source)
    - well with pollution results = check results
    - shared_tap with high queue time = partially_functional
    """
    wtype   = row['type_of_water_source']
    queue   = row['time_in_queue']  if pd.notna(row['time_in_queue'])   else 0
    poll_result = row['results']    if pd.notna(row['results'])         else 'clean'

    if wtype == 'tap_in_home':
        return 'functional'
    elif wtype == 'tap_in_home_broken':
        return 'non_functional'
    elif wtype == 'river':
        return 'non_functional'
    elif wtype == 'well':
        if poll_result in ['dirty', 
                           'Contaminated with biological',
                           'biologically contaminated']:
            return 'non_functional'
        else:
            return 'functional'
    elif wtype == 'shared_tap':
        return 'partially_functional' if queue > 30 else 'functional'
    else:
        return 'unknown'

master['functionality_status'] = master.apply(classify_status, axis=1)

# Confirm the column is there
print(master['functionality_status'].value_counts())
print(master.columns.tolist())

NameError: name 'master' is not defined

In [ ]:
master.to_csv('master_clean.csv', index=False)
print("✅ Saved with functionality_status column")

# 4. Exploratory Data Analysis (EDA)

## 4.1 Water Source Type Distribution 
####  Research Question 1 (RQ1): Which types of water points are most often non-functional?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Chart 1: Count of each water source type
water_source['type_of_water_source'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Number of Water Points by Type', fontweight='bold')
axes[0].set_xlabel('Water Source Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Chart 2: People served per type
master.groupby('type_of_water_source')['number_of_people_served'].sum().sort_values().plot(
    kind='barh', ax=axes[1], color='teal', edgecolor='white'
)
axes[1].set_title('Total People Served by Water Source Type', fontweight='bold')
axes[1].set_xlabel('Total People Served')

plt.tight_layout()
plt.savefig('chart1_source_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 Functionality Status Breakdown

In [ ]:
status_counts = master['functionality_status'].value_counts()

fig, ax = plt.subplots(figsize=(7, 5))
colors = {'functional':'#27ae60', 'partially_functional':'#f39c12', 'non_functional':'#e74c3c', 'unknown':'#95a5a6'}
bars = ax.bar(status_counts.index, status_counts.values,
              color=[colors.get(s, 'gray') for s in status_counts.index], edgecolor='white')

# Add value labels on bars
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/status_counts.sum()*100:.1f}%)', ha='center', fontsize=10)

ax.set_title('Water Point Functionality Status', fontweight='bold', fontsize=13)
ax.set_xlabel('Status'); ax.set_ylabel('Number of Records')
plt.tight_layout()
plt.savefig('chart2_functionality_status.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.3 Well Pollution Analysis (RQ2)
### # Research Question 2: What factors influence water point performance?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pollution results breakdown
well_poll['results'].value_counts().plot(
    kind='pie', ax=axes[0], autopct='%1.1f%%',
    colors=['#27ae60', '#e74c3c', '#f39c12'],
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Well Pollution Results', fontweight='bold')
axes[0].set_ylabel('')

# Pollutant ppm by result category
well_poll.boxplot(column='pollutant_ppm', by='results', ax=axes[1])
axes[1].set_title('Pollutant PPM by Result Category', fontweight='bold')
axes[1].set_xlabel('Pollution Result')
axes[1].set_ylabel('Pollutant PPM')
plt.suptitle('')

plt.tight_layout()
plt.savefig('chart4_well_pollution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Urban vs Rural Access (RQ4)
### Compare access between urban and rural

In [ ]:
# Compare access between urban and rural
urban_rural = master.groupby('location_type')['number_of_people_served'].agg(['mean','sum','count'])
print("Urban vs Rural — Water Access:")
print(urban_rural)

# Functionality by location type
func_by_loc = master.groupby(['location_type','functionality_status']).size().unstack(fill_value=0)
func_by_loc.plot(kind='bar', figsize=(9,5), edgecolor='white',
                 color=['#e74c3c','#f39c12','#27ae60','#95a5a6'])
plt.title('Functionality Status by Location Type (Urban vs Rural)', fontweight='bold')
plt.xlabel('Location Type'); plt.ylabel('Count')
plt.xticks(rotation=0); plt.legend(title='Status')
plt.tight_layout()
plt.savefig('chart5_urban_rural.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Queue Time Analysis — The Congestion Problem
### Research Question 5: How can data analysis reduce congestion at water points?

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of queue times
shared_taps = master[master['type_of_water_source'] == 'shared_tap']
axes[0].hist(shared_taps['time_in_queue'].dropna(), bins=30, color='coral', edgecolor='white')
axes[0].set_title('Queue Time Distribution (Shared Taps)', fontweight='bold')
axes[0].set_xlabel('Time in Queue (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(30, color='red', linestyle='--', label='30-min threshold')
axes[0].legend()

# Average queue time by province
avg_queue = master.groupby('province_name')['time_in_queue'].mean().sort_values(ascending=False)
avg_queue.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Average Queue Time by Province', fontweight='bold')
axes[1].set_xlabel('Average Minutes in Queue')

plt.tight_layout()
plt.savefig('chart3_queue_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nShared taps with queue > 30 min: {(shared_taps['time_in_queue'] > 30).sum()}")
print(f"% of shared taps with high queue: {(shared_taps['time_in_queue'] > 30).mean()*100:.1f}%")

## Summary Statistics Table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Total water point records',
        'Unique water sources',
        'Provinces covered',
        'Total people served',
        'Average queue time (min)',
        'Wells tested for pollution',
        'Contaminated wells (%)',
        'Non-functional water points (%)',
    ],
    'Value': [
        f"{len(master):,}",
        f"{master['source_id'].nunique():,}",
        f"{master['province_name'].nunique()}",
        f"{master['number_of_people_served'].sum():,.0f}",
        f"{master['time_in_queue'].mean():.1f}",
        f"{len(well_poll):,}",
        f"{(well_poll['results'] != 'Clean').mean()*100:.1f}%",
        f"{(master['functionality_status']=='non_functional').mean()*100:.1f}%",
    ]
})
print(summary.to_string(index=False))

# 5 Predictive Modelling
## 5.1 Prepare Features for the Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Select features (inputs) for the model
features = [
    'type_of_water_source',   # type of water point
    'number_of_people_served', # demand/load on the water point
    'time_in_queue',           # proxy for overcrowding
    'subjective_quality_score',# quality assessment score
    'location_type',           # urban vs rural
]

# Use only rows where functionality_status is known and not 'unknown'
model_df = master[master['functionality_status'] != 'unknown'][features + ['functionality_status']].copy()
model_df = model_df.dropna()

print(f"Records for modeling: {len(model_df)}")
print(model_df['functionality_status'].value_counts())

## 5.2 Encode Categorical Variables

In [ ]:
le_type = LabelEncoder()
le_loc  = LabelEncoder()
le_status = LabelEncoder()

model_df['type_encoded']   = le_type.fit_transform(model_df['type_of_water_source'])
model_df['loc_encoded']    = le_loc.fit_transform(model_df['location_type'])
model_df['status_encoded'] = le_status.fit_transform(model_df['functionality_status'])

print("Encoded classes:")
print("Source types:", list(le_type.classes_))
print("Location types:", list(le_loc.classes_))
print("Status labels:", list(le_status.classes_))

## 5.3 Split Data into Training and Testing Sets

In [ ]:
X = model_df[['type_encoded','number_of_people_served','time_in_queue',
              'subjective_quality_score','loc_encoded']]
y = model_df['status_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} records")
print(f"Testing set:  {len(X_test)} records")

## 5.4 Train the Random Forest Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)
print("Model trained successfully!")

y_pred = rf_model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {acc*100:.2f}%")

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le_status.classes_))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_status.classes_,
            yticklabels=le_status.classes_, ax=ax)
ax.set_title('Confusion Matrix — Water Point Failure Prediction', fontweight='bold')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('chart6_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.6 Feature Importance — What Drives Failure?

In [ ]:
feature_names = ['Source Type','People Served','Queue Time','Quality Score','Location Type']
importances = rf_model.feature_importances_

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feature_names, importances, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance — Which Factors Predict Failure?', fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, imp in zip(bars, importances):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{imp:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('chart7_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.7 Generate Risk Scores for Every Water Point

In [ ]:
proba = rf_model.predict_proba(X)
failure_class_index = list(le_status.classes_).index('non_functional')

model_df['failure_probability'] = proba[:, failure_class_index]
model_df['risk_level'] = pd.cut(
    model_df['failure_probability'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print("Risk level distribution:")
print(model_df['risk_level'].value_counts())

# Export with risk scores for Power BI
final_export = master.loc[model_df.index].copy()
final_export['failure_probability'] = model_df['failure_probability'].values
final_export['risk_level'] = model_df['risk_level'].values
final_export.to_csv('water_points_with_risk.csv', index=False)
print("\nExported: water_points_with_risk.csv")

In [ ]:
import pickle

# Save the model and encoders
with open('model.pkl', 'w+b') as f:
    pickle.dump(rf_model, f)

with open('encoders.pkl', 'w+b') as f:
    pickle.dump({
        'le_type':   le_type,
        'le_loc':    le_loc,
        'le_status': le_status
    }, f)

print("✅ Saved model.pkl and encoders.pkl")
print("   Copy both files into your maji_mengi_app/ folder")